# Consolidación de Métricas Técnicas (A6) y Operativas (A7)
## Comparativa multimodelo y validación de la hipótesis central (Configuración A frente a Configuración B)

Este notebook consolida los resultados de evaluación sobre la partición **test** del proyecto EvidenCite a partir de **MLflow**: cada corrida de `src/modeling` (TF-IDF, SciBERT, LLM con LoRA y modelos de API) deja allí su `metrics_summary.json` con las métricas A6 y A7 y las predicciones de test. Ya no lee `models/*/metrics_summary.json`: las corridas que llegaron completas a MLflow no dejan nada en local. Las familias y modelos que aparecen son los que haya en los experimentos elegidos.

### Objetivos analíticos
* **Hipótesis central:** ¿la Configuración B (secciones retóricas y Top-3 chunks recuperados) supera significativamente a la Configuración A (título y abstract)? Los pares A/B se arman solos: dos corridas del mismo modelo, variante, tasa, longitud y semilla que solo difieren en la configuración.
* **Rigor estadístico pareado:** intervalo de confianza del 95 % de ΔF1 por *paired bootstrap* ($B=10.000$) y prueba de *McNemar*. Las predicciones de cada corrida se emparejan instancia a instancia con el test local, después de verificar su SHA-256 contra el que registró la corrida (`sha256_test`).
* **A priori frente a final:** la sonda lineal sobre el modelo sin entrenar (`test_apriori_*`) junto al resultado tras entrenar.
* **Desglose por procedencia:** ΔF1 según la fuente de la instancia (`source_dataset`) y el origen de la etiqueta (`label_source`). Reemplaza el desglose por sección retórica: en este dataset la sección delata la fuente («Unknown Section» es casi toda MultiCite) y el EDA v3 pide reportar por procedencia.
* **Fronteras operativas y económicas:** latencia (ms) y costo proyectado en USD por volumen de producción frente al macro-F1.

**Entrada:** el servidor de MLflow en `MLFLOW_IP` (o `MLFLOW_TRACKING_URI`, o `mlflow_ip.txt`), los experimentos en `EXPERIMENTOS` (por defecto `comparacion_modelos`), un filtro opcional en `FILTRO` y el test local en `DATASET` (una carpeta con `test.jsonl` o el `dataset.zip`; por defecto `data/dataset`). Con `INCLUIR_LOCALES=1` suma las corridas que solo estén en `models/*/metrics_summary.json` (por ejemplo, de API sin servidor). Corre desde `notebooks/` con el entorno del proyecto (`uv sync`): usa `src/`.

In [ ]:
# 1. Configuración de entorno, importaciones y parámetros
import os
import sys
import json
import tempfile
import zipfile
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from mlflow import MlflowClient

# Asegurar importación de módulos del proyecto
RAIZ = Path('..') if Path('../src').exists() else Path('.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.config.schema import CANONICAL_9_CLASSES
from src.modeling.dataset import compute_split_sha256, load_jsonl_instances
from src.modeling.evaluate import (
    TechnicalMetricsA6,
    OperationalMetricsA7,
    PairedSignificanceResult,
    compute_a6_metrics,
    paired_significance_test,
)
from src.visualization import (
    plot_ab_confusion_matrices,
    plot_per_class_f1_comparison,
    plot_a6_a7_efficiency_frontier,
    plot_a7_economic_frontier,
)

%matplotlib inline


def servidor_mlflow() -> str:
    """Dirección del servidor: MLFLOW_TRACKING_URI, MLFLOW_IP o la IP escrita en mlflow_ip.txt (aquí o en el repo)."""
    candidatas = [os.environ.get("MLFLOW_TRACKING_URI", ""), os.environ.get("MLFLOW_IP", "")]
    for archivo in (Path("mlflow_ip.txt"), RAIZ / "mlflow_ip.txt"):
        if archivo.is_file():
            candidatas.append(archivo.read_text(encoding="utf-8").strip())
    for c in candidatas:
        if c.strip():
            return c.strip().rstrip("/") if "://" in c else f"http://{c.strip()}"
    raise ValueError("Define MLFLOW_IP (ip:puerto del servidor de MLflow) o escríbela en mlflow_ip.txt")


MLFLOW_URI = servidor_mlflow()
EXPERIMENTOS = [e.strip() for e in os.environ.get("EXPERIMENTOS", "comparacion_modelos").split(",") if e.strip()]
FILTRO = os.environ.get("FILTRO", "")          # filtro de MLflow, p. ej. "params.max_seq_length = '512'"
DATASET = Path(os.environ.get("DATASET", RAIZ / "data" / "dataset"))   # carpeta con test.jsonl o dataset.zip
INCLUIR_LOCALES = os.environ.get("INCLUIR_LOCALES", "0") == "1"       # suma models/*/metrics_summary.json
SOLO_VERIFICADAS = os.environ.get("SOLO_VERIFICADAS", "1") == "1"    # 0: también corridas evaluadas en otro test
N_BOOTSTRAP = int(os.environ.get("N_BOOTSTRAP", "10000"))
# Lo que deben compartir dos corridas para formar un par A/B (todo lo demás de la configuración puede variar)
CLAVES_PAR = ("model_family", "pretrained_model_name_or_path", "model_name", "lora_variant", "learning_rate",
              "max_seq_length", "seed")
CONFIG_REPRESENTATIVA = "B"   # configuración de las corridas representativas (análisis por clase)
REPRESENTATIVOS: list[str] = []   # nombres de corridas para el análisis por clase; vacío: la mejor en dev por familia
FRONTERA: list[str] = []          # corridas de las fronteras; vacío: la mejor en dev de cada familia

fig_dir = RAIZ / 'reports' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
cliente = MlflowClient(tracking_uri=MLFLOW_URI)
print(f'MLflow: {MLFLOW_URI} · experimentos: {", ".join(EXPERIMENTOS)}' + (f' · filtro: {FILTRO}' if FILTRO else ''))

## 2. Carga y validación de las corridas (MLflow → `metrics_summary.json`)

Por cada corrida terminada (`FINISHED`) de los experimentos se baja su `metrics_summary.json` y se valida contra los contratos de Pydantic (`TechnicalMetricsA6`, `OperationalMetricsA7`). Si dos corridas se llaman igual, queda la más reciente.

**Verificación del test:** las pruebas pareadas comparan predicciones instancia a instancia, así que el test local debe ser el mismo archivo con que se evaluó cada corrida. Se compara su SHA-256 con el parámetro `sha256_test` de la corrida. Las corridas sin ese parámetro (las de API o `MODO=mini`) se aceptan si tienen tantas predicciones como instancias tiene el test, con un aviso. Las que no pasan la verificación quedan fuera de todo el análisis, porque las métricas de dos versiones del test no son comparables; con `SOLO_VERIFICADAS=0` se incluyen en las tablas y fronteras, pero nunca en las pruebas pareadas ni en el desglose.

In [ ]:
def test_local(origen: Path) -> Path:
    """La ruta de test.jsonl: dentro de la carpeta `origen` o del zip `origen` (como scripts/_comun.sh)."""
    if origen.is_file() and origen.suffix == '.zip':
        destino = Path(tempfile.mkdtemp(prefix='dataset_'))
        zipfile.ZipFile(origen).extractall(destino)
        encontrados = [p for p in destino.rglob('test.jsonl') if '__MACOSX' not in p.parts]
        if not encontrados:
            raise FileNotFoundError(f'{origen} no trae test.jsonl')
        return encontrados[0]
    for carpeta in (origen, RAIZ / 'data' / 'labeled'):
        if (carpeta / 'test.jsonl').is_file():
            return carpeta / 'test.jsonl'
    raise FileNotFoundError(f'No encontré test.jsonl en {origen}: define DATASET (carpeta o dataset.zip)')


def leer_json(run_id: str, ruta: str) -> dict | None:
    """Un JSON de los artefactos de la corrida, bajado a una carpeta temporal que se borra enseguida."""
    with tempfile.TemporaryDirectory() as tmp:
        try:
            archivo = cliente.download_artifacts(run_id, ruta, tmp)
        except Exception:   # la corrida no tiene ese artefacto
            return None
        return json.loads(Path(archivo).read_text(encoding='utf-8'))


def a_corrida(nombre: str, data: dict, params: dict, origen: str, inicio: int, run_id: str | None,
              experimento: str) -> dict:
    """Valida el resumen de una corrida con los contratos A6/A7 y arma su registro."""
    previas = {split: TechnicalMetricsA6(**m) for split, m in (data.get('apriori_metrics_a6') or {}).items()}
    dev = data.get('dev_metrics_a6')
    return {
        'nombre': nombre, 'run_id': run_id, 'experimento': experimento, 'origen': origen, 'inicio': inicio,
        'params': params, 'data': data,
        'a6': TechnicalMetricsA6(**data['test_metrics_a6']),
        'a6_dev': TechnicalMetricsA6(**dev) if dev else None,
        'a7': OperationalMetricsA7(**data['operational_metrics_a7']),
        'apriori': previas,
        'predictions': data.get('test_predictions') or [],
        'family': data.get('model_family') or params.get('model_family'),
        'config': data.get('configuration') or params.get('configuration'),
        'modelo': (params.get('pretrained_model_name_or_path') or params.get('model_name') or 'TF-IDF').rstrip('/').split('/')[-1],
    }


def cargar_desde_mlflow(experimentos: list[str], filtro: str = '') -> dict[str, dict]:
    ids = {}
    for nombre in experimentos:
        exp = cliente.get_experiment_by_name(nombre)
        if exp is None:
            print(f'No existe el experimento «{nombre}» en {MLFLOW_URI}')
        else:
            ids[exp.experiment_id] = nombre
    if not ids:
        raise ValueError('Ningún experimento encontrado: revisa EXPERIMENTOS')
    filtros = ["attributes.status = 'FINISHED'"] + ([filtro] if filtro else [])
    corridas = {}
    for r in cliente.search_runs(list(ids), ' and '.join(filtros), order_by=['attributes.start_time DESC']):
        nombre = r.info.run_name
        if nombre in corridas:
            continue   # ya está la más reciente con ese nombre
        data = leer_json(r.info.run_id, 'metrics_summary.json')
        if data is None or not data.get('test_metrics_a6'):
            print(f'Salto {nombre}: no tiene metrics_summary.json con métricas de test')
            continue
        corridas[nombre] = a_corrida(nombre, data, r.data.params, 'mlflow', r.info.start_time, r.info.run_id,
                                     ids[r.info.experiment_id])
    return corridas


def cargar_locales(carpeta: Path) -> dict[str, dict]:
    """Corridas que solo están en local (models/<nombre>/metrics_summary.json), con los parámetros de su config.json."""
    corridas = {}
    for resumen in sorted(carpeta.glob('*/metrics_summary.json')):
        data = json.loads(resumen.read_text(encoding='utf-8'))
        if not data.get('test_metrics_a6'):
            continue
        params = {'model_family': data.get('model_family'), 'configuration': data.get('configuration')}
        config = resumen.parent / 'config.json'
        if config.is_file():
            cfg = json.loads(config.read_text(encoding='utf-8'))
            familia = cfg.get(cfg.get('model_family', ''), {}) or {}
            params |= {k: str(v) for k, v in familia.items() if not isinstance(v, (dict, list))}
            params |= {'seed': str(cfg.get('seed')),
                       'max_seq_length': str(cfg.get('input_representation', {}).get('max_seq_length'))}
        corridas[resumen.parent.name] = a_corrida(resumen.parent.name, data, params, 'local',
                                                  int(resumen.stat().st_mtime * 1000), None, 'local')
    return corridas


ruta_test = test_local(DATASET)
test_instances = load_jsonl_instances(ruta_test)
y_true = [inst.label for inst in test_instances]
hash_test = compute_split_sha256(ruta_test)
print(f'Test local: {ruta_test} · {len(y_true)} instancias · sha256 {hash_test[:12]}…')

corridas = cargar_desde_mlflow(EXPERIMENTOS, FILTRO)
if INCLUIR_LOCALES:
    for nombre, c in cargar_locales(RAIZ / 'models').items():
        corridas.setdefault(nombre, c)

for c in corridas.values():   # ¿sus predicciones se pueden emparejar con este test?
    registrado = c['params'].get('sha256_test')
    if registrado:
        c['verificada'] = registrado == hash_test and len(c['predictions']) == len(y_true)
        c['nota'] = 'test verificado' if c['verificada'] else 'otro test (sha256 distinto)'
    else:
        c['verificada'] = len(c['predictions']) == len(y_true)
        c['nota'] = ('sin sha256_test: se asume este test' if c['verificada']
                     else f"sin sha256_test y {len(c['predictions'])} predicciones")

print(f'\nCorridas cargadas y validadas ({len(corridas)}):')
for name, c in sorted(corridas.items()):
    print(f" • {name:<58} | {c['family']:<8} | Config {c['config']} | Macro-F1 test {c['a6'].macro_f1_all_9:.4f} | {c['nota']}")
if SOLO_VERIFICADAS:   # métricas de tests distintos no son comparables: quedan solo las de este test
    fuera = sorted(n for n, c in corridas.items() if not c['verificada'])
    corridas = {n: c for n, c in corridas.items() if c['verificada']}
    if fuera:
        print(f'\nQuedan fuera {len(fuera)} corridas evaluadas en otro test (SOLO_VERIFICADAS=0 las incluye en las '
              f'tablas y fronteras): {", ".join(fuera)}')
# Rótulo de cada corrida en tablas y figuras: «modelo (config)»; si se repite, con la familia; si aún, su nombre
for c in corridas.values():
    c['etiqueta'] = f"{c['modelo']} ({c['config']})"
for paso in ('familia', 'nombre'):
    vistos = [c['etiqueta'] for c in corridas.values()]
    for c in corridas.values():
        if vistos.count(c['etiqueta']) > 1:
            c['etiqueta'] = f"{c['modelo']} · {c['family']} ({c['config']})" if paso == 'familia' else c['nombre']
print(f'\n{len(corridas)} corridas en el análisis.')

## 3. Tabla resumen multidimensional (A6 y A7)

Una fila por corrida: métricas técnicas de test (macro-F1 de las 9 clases y accuracy), la sonda lineal a priori (el macro-F1 de test del modelo sin entrenar; «—» si la corrida no la tiene) y las métricas operativas (latencia, throughput, parámetros entrenables y costo).

In [ ]:
filas_tabla = []
for name, c in corridas.items():
    a6, a7 = c['a6'], c['a7']
    filas_tabla.append({
        'Experimento': name,
        'Familia': c['family'],
        'Modelo': c['modelo'],
        'Config': c['config'],
        'Macro-F1 (9)': a6.macro_f1_all_9,
        'Macro-F1 a priori': c['apriori']['test'].macro_f1_all_9 if 'test' in c['apriori'] else np.nan,
        'Accuracy (Micro-F1)': a6.micro_f1,
        'Latencia (ms/ej)': a7.inference_latency_ms_per_instance,
        'Throughput (ej/s)': a7.throughput_instances_per_sec,
        'Parámetros Entrenables (%)': a7.trainable_ratio_pct,
        'Costo Est. (USD)': a7.estimated_api_cost_usd,
    })

df_resumen = pd.DataFrame(filas_tabla).sort_values(by=['Familia', 'Modelo', 'Config']).reset_index(drop=True)
styled_table = df_resumen.style.format({
    'Macro-F1 (9)': '{:.4f}',
    'Macro-F1 a priori': '{:.4f}',
    'Accuracy (Micro-F1)': '{:.4f}',
    'Latencia (ms/ej)': '{:.2f}',
    'Throughput (ej/s)': '{:.1f}',
    'Parámetros Entrenables (%)': '{:.2f}%',
    'Costo Est. (USD)': '${:.2f}',
}, na_rep='—').hide(axis='index').set_caption(f'Tabla 1: Comparativa multidimensional de desempeño y costo operativo (test, N={len(y_true):,})')

styled_table

## 4. Validación de la hipótesis central: Configuración A frente a Configuración B

### Pares automáticos, significancia pareada y matrices de confusión
Dos corridas forman un par si comparten `CLAVES_PAR` (familia, modelo, variante, tasa, longitud y semilla) y una es A y la otra B; si hay varias, se usa la más reciente de cada configuración. Para cada par:
1. **Prueba de significancia pareada:** *paired bootstrap* con $B=10.000$ réplicas y prueba de McNemar, sobre las predicciones de test emparejadas por instancia.
2. **Matriz de confusión triple ($A$, $B$ y diferencia $B - A$):** con `plot_ab_confusion_matrices`. En el panel de diferencia:
   * **Valores verdes positivos en la diagonal:** clases donde B aumentó el recall frente a A.
   * **Valores rojos fuera de la diagonal:** confusiones que B redujo o eliminó.

Solo entran las corridas con el test verificado (sección 2).

In [ ]:
def clave_par(c: dict) -> tuple:
    return tuple(c['params'].get(k) for k in CLAVES_PAR)


grupos: dict[tuple, dict[str, dict]] = {}
for c in sorted(corridas.values(), key=lambda c: c['inicio']):   # la más reciente pisa a las anteriores
    if c['config'] in ('A', 'B') and c['verificada']:
        grupos.setdefault(clave_par(c), {})[c['config']] = c

pares = []
for clave, por_config in grupos.items():
    if {'A', 'B'} <= por_config.keys():
        c_a, c_b = por_config['A'], por_config['B']
        variante = c_a['params'].get('lora_variant')
        titulo = c_a['modelo'] + (f' · {variante}' if variante not in (None, 'lora') else '')
        pares.append((titulo, c_a['nombre'], c_b['nombre']))
# Si un título se repite (otra familia, otra tasa u otra semilla), lleva la familia y la tasa para distinguirlos
titulos = [t for t, _, _ in pares]
pares = [(f"{t} · {corridas[a]['family']} · lr {corridas[a]['params'].get('learning_rate')}" if titulos.count(t) > 1
          else t, a, b) for t, a, b in pares]
print(f'{len(pares)} pares A/B:')
for titulo, exp_a, exp_b in pares:
    print(f' • {titulo:<40} A: {exp_a} · B: {exp_b}')

resultados_significancia = []
for titulo_fam, exp_a, exp_b in pares:
    c_a, c_b = corridas[exp_a], corridas[exp_b]
    sig: PairedSignificanceResult = paired_significance_test(
        y_true=y_true,
        y_pred_a=c_a['predictions'],
        y_pred_b=c_b['predictions'],
        n_bootstrap=N_BOOTSTRAP,
        seed=42,
    )
    resultados_significancia.append({
        'Par': titulo_fam,
        'Macro-F1 (A)': sig.score_a,
        'Macro-F1 (B)': sig.score_b,
        'Δ (B - A)': sig.delta_b_minus_a,
        'IC 95% Δ [Inf, Sup]': f'[{sig.ci_95_low:+.4f}, {sig.ci_95_high:+.4f}]',
        'p-val (Bootstrap)': sig.p_value_bootstrap,
        'p-val (McNemar)': sig.p_value_mcnemar,
        'Significativo (p < 0.05)': 'SÍ' if sig.is_significant_95 else 'NO',
    })

    nombre_fig = ''.join(ch if ch.isalnum() or ch in '-._' else '_' for ch in titulo_fam)
    fig = plot_ab_confusion_matrices(
        eval_a=c_a['a6'],
        eval_b=c_b['a6'],
        class_names=CANONICAL_9_CLASSES,
        output_path=fig_dir / f'confusion_differential_{nombre_fig}.png',
    )
    fig.suptitle(f'{titulo_fam} · Matrices de confusión A vs. B y panel diferencial (B - A)', fontsize=13, fontweight='bold', y=1.02)
    plt.show()

df_sig = pd.DataFrame(resultados_significancia)
print('\n=== RESUMEN FORMAL DE SIGNIFICANCIA ESTADÍSTICA (PAIRED BOOTSTRAP + MCNEMAR) ===')
df_sig.style.format({
    'Macro-F1 (A)': '{:.4f}',
    'Macro-F1 (B)': '{:.4f}',
    'Δ (B - A)': '{:+.4f}',
    'p-val (Bootstrap)': '{:.4e}',
    'p-val (McNemar)': '{:.4e}',
}) if not df_sig.empty else print('Sin pares A/B completos y verificados.')

## 5. Análisis granular por clase canónica (A6)

F1 de cada una de las 9 clases para las corridas representativas en la **Configuración B** (`CONFIG_REPRESENTATIVA`): por defecto, la mejor corrida de cada familia según el macro-F1 de dev (las de API, sin dev, por test). Se pueden fijar a mano en `REPRESENTATIVOS`. La tabla agrega, si la corrida la tiene, el F1 por clase de la sonda lineal a priori.

In [ ]:
def criterio(c: dict) -> float:
    """El macro-F1 de dev (el que sirve para elegir); las corridas sin dev (API) usan el de test."""
    return (c['a6_dev'] or c['a6']).macro_f1_all_9


def etiqueta(c: dict) -> str:
    return c['etiqueta']


if REPRESENTATIVOS:
    representativas = [corridas[n] for n in REPRESENTATIVOS]
else:
    mejores = {}
    for c in corridas.values():
        if c['config'] == CONFIG_REPRESENTATIVA and (c['family'] not in mejores or criterio(c) > criterio(mejores[c['family']])):
            mejores[c['family']] = c
    representativas = sorted(mejores.values(), key=criterio, reverse=True)[:5]   # la figura tiene 5 colores

if representativas:
    comparativa_por_clase = {etiqueta(c): c['a6'] for c in representativas}
    fig_barras_path = fig_dir / f'f1_por_clase_comparativa_config_{CONFIG_REPRESENTATIVA.lower()}.png'
    fig_f1 = plot_per_class_f1_comparison(
        results_by_config=comparativa_por_clase,
        output_path=fig_barras_path,
        title=f'Desempeño F1-Score por intención de cita canónica (Configuración {CONFIG_REPRESENTATIVA})',
    )
    plt.show()
    print(f'Figura guardada en: {fig_barras_path}')
    columnas = {}
    for c in representativas:   # cada corrida y, a su derecha, su sonda a priori
        columnas[etiqueta(c)] = c['a6'].per_class_f1
        if 'test' in c['apriori']:
            columnas[f'{etiqueta(c)} · a priori'] = c['apriori']['test'].per_class_f1
    por_clase = pd.DataFrame(columnas)
    display(por_clase.reindex(CANONICAL_9_CLASSES).style.format('{:.3f}', na_rep='—')
            .set_caption('F1 de test por clase (y de la sonda lineal a priori, si la hay)'))
else:
    print(f'No hay corridas en la Configuración {CONFIG_REPRESENTATIVA}.')

## 6. Desglose por procedencia

Macro-F1 (sobre las clases presentes en cada grupo) según la **fuente** de la instancia (`source_dataset`) y el **origen de la etiqueta** (`label_source`): humanas, de regla o de NLI. Primero el nivel de las corridas representativas; después, por cada par A/B, la diferencia B − A en cada grupo. Un grupo pequeño (columna `n`) da diferencias ruidosas. Para otro corte, agrega un campo a `DESGLOSES` (por ejemplo `'rhetorical_section'`).

In [ ]:
DESGLOSES = {
    'source_dataset': lambda inst: inst.annotation_metadata.source_dataset,
    'label_source': lambda inst: inst.annotation_metadata.label_source,
    # 'rhetorical_section': lambda inst: inst.citing_paper.rhetorical_section,
}


def f1_por_grupo(predicciones: list[str], grupos: np.ndarray) -> pd.Series:
    """Macro-F1 de las clases presentes en cada grupo de instancias."""
    verdad, pred = np.asarray(y_true), np.asarray(predicciones)
    return pd.Series({g: compute_a6_metrics(verdad[grupos == g].tolist(), pred[grupos == g].tolist()).macro_f1_present
                      for g in pd.unique(grupos)})


for campo, extraer in DESGLOSES.items():
    grupos_inst = np.array([extraer(inst) or '—' for inst in test_instances], dtype=object)
    conteo = pd.Series(grupos_inst).value_counts().rename('n')
    nivel = pd.DataFrame({etiqueta(c): f1_por_grupo(c['predictions'], grupos_inst)
                          for c in representativas if c['verificada']})
    if not nivel.empty:
        display(pd.concat([conteo, nivel], axis=1).sort_values('n', ascending=False)
                .style.format('{:.3f}', subset=list(nivel.columns)).set_caption(f'Macro-F1 por {campo} (corridas representativas)'))
    deltas = pd.DataFrame({titulo: f1_por_grupo(corridas[b]['predictions'], grupos_inst)
                                   - f1_por_grupo(corridas[a]['predictions'], grupos_inst)
                           for titulo, a, b in pares})
    if not deltas.empty:
        display(pd.concat([conteo, deltas], axis=1).sort_values('n', ascending=False)
                .style.format('{:+.3f}', subset=list(deltas.columns)).set_caption(f'Δ macro-F1 (B − A) por {campo}'))

## 7. Fronteras operativas y económicas (A7)

### 7.a. Frontera técnica de latencia frente a macro-F1
Compromiso entre **calidad predictiva (macro-F1, eje Y)** y **latencia de inferencia (ms por cita, eje X en escala logarítmica)**. El tamaño de cada burbuja refleja el total de parámetros de la arquitectura y las APIs llevan rombo. Por defecto entra la mejor corrida de cada familia según dev, como en la versión anterior (con varios LLM, sus burbujas caen casi en el mismo punto y los rótulos se pisan); `FRONTERA` permite elegir las corridas.

In [ ]:
if FRONTERA:
    seleccion = [corridas[n] for n in FRONTERA]
else:
    mejor_por_familia = {}   # una burbuja por familia: los LLM se amontonan en latencia y F1 y sus rótulos se pisan
    for c in corridas.values():
        if c['family'] not in mejor_por_familia or criterio(c) > criterio(mejor_por_familia[c['family']]):
            mejor_por_familia[c['family']] = c
    seleccion = sorted(mejor_por_familia.values(), key=criterio, reverse=True)
modelos_para_frontera = [(etiqueta(c), c['a6'], c['a7']) for c in seleccion]

fig_frontera_path = fig_dir / 'frontera_eficiencia_a6_a7.png'
fig_frontera = plot_a6_a7_efficiency_frontier(
    records=modelos_para_frontera,
    output_path=fig_frontera_path,
    title='Frontera de eficiencia de Pareto: latencia (ms) vs. macro-F1 (9 clases)',
)
plt.show()
print(f'Figura de frontera de eficiencia guardada en: {fig_frontera_path}')

### 7.b. Frontera económica A7 (costo en USD por 10.000 citas frente a macro-F1)
Viabilidad económica en producción para un volumen de 10.000 citas: el costo marginal casi nulo de los modelos locales frente al costo por token de las APIs comerciales.

In [ ]:
fig_econ_path = fig_dir / 'frontera_economica_a7_10k_citas.png'
fig_econ = plot_a7_economic_frontier(
    records=modelos_para_frontera,
    target_citations=10000,
    test_set_size=len(y_true),
    output_path=fig_econ_path,
    title='Frontera económica A7: costo financiero (USD por 10.000 citas) vs. macro-F1',
)
plt.show()
print(f'Figura de frontera económica guardada en: {fig_econ_path}')

## 8. Conclusiones metodológicas y respuesta a las preguntas de investigación

La celda siguiente redacta el resumen a partir de las corridas cargadas (la versión anterior traía números fijos de demostración). Úsalo como base para la discusión: no interpreta por qué gana una configuración ni reemplaza el análisis de errores.

In [ ]:
def rho_spearman(x, y) -> float:
    rx, ry = pd.Series(x).rank(), pd.Series(y).rank()
    return float(np.corrcoef(rx, ry)[0, 1])


lineas = []
con_dev = [c for c in corridas.values() if c['a6_dev'] is not None]
if con_dev:
    mejor = max(con_dev, key=criterio)
    lineas.append(f"1. **Mejor corrida según dev:** {etiqueta(mejor)} (`{mejor['nombre']}`), macro-F1 de dev "
                  f"{mejor['a6_dev'].macro_f1_all_9:.3f} y de test {mejor['a6'].macro_f1_all_9:.3f}.")
if resultados_significancia:
    significativos = [r for r in resultados_significancia if r['Significativo (p < 0.05)'] == 'SÍ']
    a_favor = [r['Par'] for r in significativos if r['Δ (B - A)'] > 0]
    en_contra = [r['Par'] for r in significativos if r['Δ (B - A)'] < 0]
    media = np.mean([r['Δ (B - A)'] for r in resultados_significancia])
    lineas.append(f"2. **Configuración B frente a A:** {len(pares)} pares; Δ medio (B − A) {media:+.4f}. "
                  f"B supera a A con p < 0.05 en {len(a_favor)} ({', '.join(a_favor) or 'ninguno'}) y queda por debajo "
                  f"con p < 0.05 en {len(en_contra)} ({', '.join(en_contra) or 'ninguno'}); en el resto la diferencia "
                  f"no es significativa.")
previas = [(c['apriori']['test'].macro_f1_all_9, c['a6'].macro_f1_all_9) for c in corridas.values()
           if 'test' in c['apriori']]
if len(previas) >= 3:
    x, y = zip(*previas)
    ganancia = np.mean(np.subtract(y, x))
    lineas.append(f"3. **A priori frente a final:** {len(previas)} corridas con sonda lineal; entrenar suma en promedio "
                  f"{ganancia:+.3f} de macro-F1 de test y el orden a priori se correlaciona con el final con ρ de "
                  f"Spearman {rho_spearman(x, y):.2f} (el detalle está en analizar_resultados.ipynb).")
if seleccion:
    rapido = min(seleccion, key=lambda c: c['a7'].inference_latency_ms_per_instance)
    lineas.append(f"4. **Frontera operativa:** la corrida más rápida de la frontera es {etiqueta(rapido)} "
                  f"({rapido['a7'].inference_latency_ms_per_instance:.2f} ms por cita, macro-F1 de test "
                  f"{rapido['a6'].macro_f1_all_9:.3f}); la de mejor macro-F1 de dev tarda "
                  f"{seleccion[0]['a7'].inference_latency_ms_per_instance:.2f} ms. Las APIs suman costo por token "
                  f"(sección 7.b).")
display(Markdown('\n'.join(lineas) if lineas else 'Sin corridas suficientes para resumir.'))